# Step 4: Main deep model — feature-enhanced attention ranker with ID masking

**Why a deep model, and why this one.** Step 3b showed Group 10's attention network reads content well but has no behavioural evidence: it was the strongest model on cold articles (test Cold Recall@10 0.574) yet weaker overall than a one-line click-rate rule (nDCG@10 0.344 vs 0.357). The simple baselines show the reverse. This model keeps Group 10's network and adds exactly what it was missing, so both kinds of evidence meet in one network.

```
article text + entities ─► news encoder (attention, Group 10) ─────┬─ candidate vector ─┐
user's last 10 clicks ───► same news encoder ─► user encoder ──────┴─ user vector ──────┤
      └─► candidate-aware attention (candidate = query) ─► user vector for this candidate ┤
                    that vector ⊙ candidate vector (GMF term) ─────────────────────────┤
engineered features (T1, T2, T3, U) ─► feature tower (2 dense layers) ──────────────────┼─► scoring MLP ─► score
article ID ─► ID embedding, masked to [UNKNOWN] at random during training ──────────────┤   (Group 10's, widened)
Group 10's user features, topics, hour, habit, recency ─────────────────────────────────┘
```

| Component | From | Course topic |
|---|---|---|
| News and user encoders (multi-head self-attention, additive attention) | Group 10, unchanged | Topic 7B: attention, Transformer blocks |
| Candidate-aware attention over the history (the candidate is the query) | New | Topic 7B: attention (query, key, value) |
| Feature tower, late fusion into the scorer | New | Topic 6: NMF with features (late fusion) |
| GMF term (element-wise user × article) | New | Topic 6: NeuMF |
| Article ID embedding with **ID masking** | New | Topic 6: cold start, "ID masking (embedding dropout)" |
| Entity vectors from the WikiData knowledge graph | Group 10 | Topic 9: knowledge-graph signals |
| One click vs four sampled non-clicks, softmax cross-entropy | Group 10 | Topic 6: negative sampling |

**Candidate-aware attention.** Group 10's user encoder builds one user vector before it sees the candidate. In our first run its attention collapsed: about 81% of the weight landed on a single past click, and that click matched the clicked article's category no more often than a random past click (20.7% vs 20.2%). The new branch lets each candidate act as the attention query, so a football article can look at the user's football clicks. Its scoring weight starts near zero, so its attention starts uniform rather than all-or-nothing. Group 10's user vector is kept as it was; step 5 retrains without the new branch to measure it.

**ID masking.** An ID embedding lets the network memorise individual articles, which helps warm articles but tempts the network to ignore content ("lazy" ID look-ups, Topic 6). During training each candidate's ID vector is replaced by a shared [UNKNOWN] vector with probability 0.3, so the network must also rank articles from content and behaviour. At scoring time every article that never appeared in a training sample gets [UNKNOWN], which is what a genuinely new article looks like. Step 5 retrains without masking to measure its effect.

**Features.** The same engineered features as the LightGBM comparison (feature schema contract v0.1), except the title/entity PCs: the news encoder already reads the full vectors. Count-like columns are log-transformed; all are standardised with training statistics.

**Training and selection.** Group 10's loop and hyperparameters (Adam, lr 5×10⁻⁴, batch 32, up to 10 epochs), seeds fixed, early stopping after 3 epochs without gain, and the epoch with the best validation nDCG@10 restored before anything else is scored. Test is scored once.

**Change from the interim plan.** The interim report planned a click-probability model; the main model is now this deep ranker. LightGBM (step 3c) is kept as the strongest non-deep comparison (Topic 11, gradient boosting).

> Runs on Colab GPU or CPU. Metrics here come from `shared/eval_interim.py`; final numbers come from `step6_evaluation_diagnostics.ipynb`.

**At a glance**

| | |
|---|---|
| Reads | `features/v0.1/`, `nrms_cache/real/` (from step 3b) and other models' `predictions/real/` on Google Drive `MyDrive/BT4222_G11/`; MIND-small via `kagglehub` |
| Writes | `predictions/real/` (`deep_hybrid`); `models/real/deep_hybrid_full.pt` and its training history |
| Time and hardware | About 25 minutes on CPU; faster on a Colab T4 GPU |
| Run after | Steps 3a, 3b and 3c (3b writes the text-vector cache; all three feed the comparison) |

## 1. Setup

In [1]:
import os, sys, subprocess, json, time

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Bryng23/BT4222-Group-11.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/BT4222-Group-11"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
    DATA_ROOT = "/content/drive/MyDrive/BT4222_G11"
else:
    REPO_DIR = os.environ.get("G11_REPO_DIR", os.getcwd())
    DATA_ROOT = os.environ.get("G11_DATA_ROOT", os.path.join(REPO_DIR, "data"))
sys.path.insert(0, REPO_DIR)

import numpy as np
import pandas as pd
import torch
from shared import schema as S, selection as SEL
from shared.predictions import write_predictions, read_predictions
from shared.nrms_inputs import build_inputs
from shared import nrms_g10 as N, hybrid_model as H
from shared import eval_interim as E      # swap for the team evaluator once it exists

DATA_KIND = "REAL"   # recorded in every prediction file
import kagglehub
RAW_DIR = os.path.join(kagglehub.dataset_download("arashnic/mind-news-dataset"), "MINDsmall_train")
FEATURE_ROOT = os.path.join(DATA_ROOT, "features", f"v{S.SCHEMA_VERSION}")
try:
    import spacy; spacy.load("en_core_web_sm")
except OSError:
    subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=True)
PRED_ROOT = os.path.join(DATA_ROOT, "predictions")
CACHE_DIR = os.path.join(DATA_ROOT, "nrms_cache", DATA_KIND.lower())   # MiniLM / spaCy vectors from step 3b
MODEL_DIR = os.path.join(DATA_ROOT, "models", DATA_KIND.lower())
os.makedirs(MODEL_DIR, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 4222

G10_CONFIG = dict(num_heads=2, dropout=0.1, user_heads=2, user_dropout=0.1)   # Group 10 Test_2, as step 3b
TRAIN = dict(epochs=10, lr=5e-4, batch_size=32, patience=3)                    # Group 10's loop + early stop
SELECT = "ndcg@10"           # checkpoint criterion: best validation nDCG@10
ALPHA = 20                   # content_smoothed_ctr strength for the +H variant (step 3a's validation choice)
SHOW = ["auc", "mrr", "ndcg@5", "ndcg@10", "cold_recall@10", "cold_hit@10", "warm_recall@10"]
print(f"DATA_KIND={DATA_KIND} | device={DEVICE} | features: {FEATURE_ROOT}")

C:\Users\Bryan Ng\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DATA_KIND=REAL | device=cpu | features: C:\Users\Bryan Ng\g11_data\features\v0.1


## 2. Model inputs

Rebuilt from the raw MIND files with Group 10's recipes (see step 3b); the MiniLM and spaCy vectors are cached, so this is quick after step 3b has run once. Candidate rows come from the shared metadata tables, so every score carries the shared `row_id`.

In [2]:
t0 = time.time()
META_COLS = ["row_id", "impression_id", "news_id", "time", "label", "is_cold_10"]
meta = {s: pd.read_parquet(S.meta_path(FEATURE_ROOT, s), columns=META_COLS) for s in S.SPLITS}
inputs = build_inputs(RAW_DIR, meta, device=DEVICE, cache_dir=CACHE_DIR, seed=SEED)
for split in S.SPLITS:
    assert (inputs["impressions"]["impression_id"][inputs["splits"][split]["imp"]] == meta[split]["impression_id"].to_numpy()).all()
print(f"inputs built in {time.time() - t0:.0f}s; {inputs['info']['n_train_samples']:,} training samples "
      f"(1 click + 4 non-clicks each), {len(inputs['news']['title_abs']):,} articles")

inputs built in 224s; 141,558 training samples (1 click + 4 non-clicks each), 51,282 articles


## 3. Feature loading and one training run

`run_variant` is shared with step 5: it loads the feature groups a variant uses, trains from scratch and restores the best validation epoch.

In [3]:
def load_features(groups):
    """Scaled feature rows for every split, aligned with meta on row_id. Scaling is fitted on train only."""
    num_cols, cat_cols = H.variant_columns(groups)
    raw = {s: pd.read_parquet(S.features_path(FEATURE_ROOT, s), columns=["row_id"] + H.raw_columns(num_cols, cat_cols))
           for s in S.SPLITS}
    for s in S.SPLITS:
        assert np.array_equal(raw[s]["row_id"].to_numpy(), meta[s]["row_id"].to_numpy()), f"{s}: row_id misaligned"
    scaler = H.FeatureScaler(num_cols, cat_cols, alpha=ALPHA).fit(raw["train"])
    feats = {s: scaler.transform(raw[s]) for s in S.SPLITS}
    cat_sizes = [(int(max(raw[s][c].max() for s in S.SPLITS)) + 1, H.CAT_DIMS[c]) for c in cat_cols]
    return feats, num_cols, cat_cols, cat_sizes


def run_variant(name, score_splits=("val", "test"), log=print):
    """Train one variant from scratch, restore its best validation epoch, score the requested splits."""
    spec = H.VARIANTS[name]
    feats, num_cols, cat_cols, cat_sizes = load_features(spec["groups"])
    N.set_seed(SEED)
    model = H.HybridNRMS(inputs, len(num_cols), cat_sizes, id_mask_p=spec["id_mask_p"],
                         candidate_attention=spec.get("candidate_attention", True), **G10_CONFIG).to(DEVICE)
    val_meta, val_split = meta["val"], inputs["splits"]["val"]

    def val_fn(m):
        res = E.evaluate(val_meta, H.score_split(m, val_split, *feats["val"]))
        return {k: res[k] for k in ["auc", "mrr", "ndcg@5", "ndcg@10", "cold_recall@10"]}

    t0 = time.time()
    history, best_epoch = H.train(model, inputs["train_samples"], *feats["train"], seed=SEED, val_fn=val_fn,
                                  select_metric=SELECT, log=log, **TRAIN)
    scores = {s: H.score_split(model, inputs["splits"][s], *feats[s]) for s in score_splits}
    return {"model": model, "history": history, "best_epoch": best_epoch, "scores": scores, "num_cols": num_cols,
            "cat_cols": cat_cols, "minutes": round((time.time() - t0) / 60, 1)}

## 4. Train the main model

Validation metrics are printed after every epoch; the checkpoint with the best validation nDCG@10 is kept.

In [4]:
res = run_variant("full", score_splits=S.SPLITS)
model = res["model"]
print(f"best validation {SELECT} at epoch {res['best_epoch']}; trained in {res['minutes']} min; "
      f"{len(res['num_cols'])} numeric + {len(res['cat_cols'])} categorical feature columns")
torch.save(model.state_dict(), os.path.join(MODEL_DIR, "deep_hybrid_full.pt"))
res["history"].to_csv(os.path.join(MODEL_DIR, "deep_hybrid_full_history.csv"))
res["history"].round(4)

C:\Users\Bryan Ng\OneDrive\Documents\NUS Work\BT4222\Project\BT4222-Group-11\shared\hybrid_model.py:300: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  torch.as_tensor(num[sl], device=dev), torch.as_tensor(cat[sl], device=dev),


epoch=1, train_loss=1.3103, train_seconds=160.2000, val_auc=0.7120, val_mrr=0.3570, val_ndcg@5=0.3887, val_ndcg@10=0.4468, val_cold_recall@10=0.6111


epoch=2, train_loss=1.2807, train_seconds=175.2000, val_auc=0.7140, val_mrr=0.3590, val_ndcg@5=0.3916, val_ndcg@10=0.4494, val_cold_recall@10=0.7185


epoch=3, train_loss=1.2713, train_seconds=169.9000, val_auc=0.7163, val_mrr=0.3606, val_ndcg@5=0.3930, val_ndcg@10=0.4509, val_cold_recall@10=0.6875


epoch=4, train_loss=1.2649, train_seconds=180.5000, val_auc=0.7120, val_mrr=0.3583, val_ndcg@5=0.3903, val_ndcg@10=0.4479, val_cold_recall@10=0.6092


epoch=5, train_loss=1.2581, train_seconds=170.4000, val_auc=0.7103, val_mrr=0.3545, val_ndcg@5=0.3858, val_ndcg@10=0.4448, val_cold_recall@10=0.5975


epoch=6, train_loss=1.2531, train_seconds=177.7000, val_auc=0.7077, val_mrr=0.3544, val_ndcg@5=0.3856, val_ndcg@10=0.4435, val_cold_recall@10=0.6994
stopping: no validation gain for 3 epochs


best validation ndcg@10 at epoch 3; trained in 19.0 min; 24 numeric + 2 categorical feature columns


,train_loss,train_seconds,val_auc,val_mrr,val_ndcg@5,val_ndcg@10,val_cold_recall@10
epoch,,,,,,,
1,1.3103,160.2,0.7120,0.3570,0.3887,0.4468,0.6111
2,1.2807,175.2,0.7140,0.3590,0.3916,0.4494,0.7185
3,1.2713,169.9,0.7163,0.3606,0.3930,0.4509,0.6875
4,1.2649,180.5,0.7120,0.3583,0.3903,0.4479,0.6092
5,1.2581,170.4,0.7103,0.3545,0.3858,0.4448,0.5975
6,1.2531,177.7,0.7077,0.3544,0.3856,0.4435,0.6994


## 5. Score every split with the selected epoch

Train, validation and test on full slates, written as prediction files. This is the only time the test split is scored.

In [5]:
CONFIG_ID = f"full,id_mask=0.3,epoch={res['best_epoch']}"
final = {}
for split in S.SPLITS:
    write_predictions(meta[split]["row_id"], res["scores"][split], root=PRED_ROOT, data_kind=DATA_KIND,
                      model_id="deep_hybrid", config_id=CONFIG_ID, split=split, schema_version=S.SCHEMA_VERSION,
                      extra={"selected_on": "val", "selection_metric": SELECT, "variant": "full"})
    final[split] = E.evaluate(meta[split], res["scores"][split])
pd.DataFrame(final).T[SHOW + ["n_cold_click_impressions"]].round(4)

,auc,mrr,ndcg@5,ndcg@10,cold_recall@10,cold_hit@10,warm_recall@10,n_cold_click_impressions
train,0.7557,0.4183,0.4610,0.5141,0.6672,0.6857,0.7441,3223.0
val,0.7163,0.3606,0.3930,0.4509,0.6875,0.6972,0.6856,938.0
test,0.7095,0.3366,0.3672,0.4279,0.6268,0.6464,0.6660,837.0


## 6. Comparison with every baseline and with LightGBM

All models are evaluated here on the same rows with the same metric code, from their prediction files. The paired bootstrap (1,000 resamples over impressions) gives the deep model's gain over Group 10's network and over LightGBM with a 95% interval.

In [6]:
import glob
COMPARE = ["deep_hybrid", "lgbm_full", "lgbm_full_binary", "nrms_g10", "content_smoothed_ctr", "candidate_smoothed_ctr",
           "category_ctr", "popularity", "random"]
scores_by = {}
for split in ("val", "test"):
    for path in sorted(glob.glob(os.path.join(PRED_ROOT, DATA_KIND.lower(), f"{DATA_KIND}__*__{split}.parquet"))):
        p, info = read_predictions(path)
        if info.get("selected_on") != "val" or info["model_id"] not in COMPARE:
            continue
        sc = meta[split][["row_id"]].merge(p, on="row_id", how="left")["predicted_score"]
        if not sc.isna().any():
            scores_by[(split, info["model_id"])] = sc.to_numpy()
rows = [{"split": s, "model": m, **{k: E.evaluate(meta[s], sc)[k] for k in SHOW}} for (s, m), sc in scores_by.items()]
table = pd.DataFrame(rows).set_index(["split", "model"])
table = table.loc[[(s, m) for s in ("val", "test") for m in COMPARE if (s, m) in table.index]]

gain_rows = []
for split in ("val", "test"):
    mine = E.per_impression_metrics(meta[split], scores_by[(split, "deep_hybrid")])
    for other in ("nrms_g10", "lgbm_full"):
        if (split, other) not in scores_by:
            continue
        theirs = E.per_impression_metrics(meta[split], scores_by[(split, other)])
        for metric in ("ndcg@10", "cold_recall@10"):
            bs = E.paired_bootstrap(theirs[metric], mine[metric])
            gain_rows.append({"split": split, "vs": other, "metric": metric, "deep_hybrid − other": bs["mean_diff"],
                              "95% CI": f"[{bs['ci_low']:+.4f}, {bs['ci_high']:+.4f}]"})
from IPython.display import display
display(table.round(4))
pd.DataFrame(gain_rows).set_index(["split", "vs", "metric"]).round(4)

auc     mrr  ndcg@5  ndcg@10  cold_recall@10  \
split model                                                                     
val   deep_hybrid             0.7163  0.3606  0.3930   0.4509          0.6875   
      lgbm_full               0.7166  0.3630  0.3968   0.4533          0.6507   
      lgbm_full_binary        0.7136  0.3558  0.3889   0.4468          0.5570   
      nrms_g10                0.6365  0.3018  0.3254   0.3829          0.5803   
      content_smoothed_ctr    0.6449  0.2966  0.3175   0.3769          0.3701   
      candidate_smoothed_ctr  0.6434  0.2975  0.3183   0.3773          0.2932   
      category_ctr            0.4903  0.2085  0.2154   0.2708          0.3324   
      popularity              0.4836  0.2192  0.2212   0.2755          0.0309   
      random                  0.5000  0.2182  0.2213   0.2779          0.2725   
test  deep_hybrid             0.7095  0.3366  0.3672   0.4279          0.6268   
      lgbm_full               0.7141  0.3376  0.3695   0.4306          0.6054   
      lgbm_full_binary        0.7102  0.3341  0.3631   0.4257          0.4932   
      nrms_g10                0.6094  0.2658  0.2815   0.3439          0.5743   
      content_smoothed_ctr    0.6479  0.2775  0.2929   0.3563          0.2694   
      candidate_smoothed_ctr  0.6468  0.2784  0.2937   0.3569          0.2489   
      category_ctr            0.4977  0.1945  0.1929   0.2549          0.2979   
      popularity              0.4608  0.1822  0.1792   0.2405          0.0287   
      random                  0.5000  0.2026  0.2034   0.2623          0.2282   

                              cold_hit@10  warm_recall@10  
split model                                                
val   deep_hybrid                  0.6972          0.6856  
      lgbm_full                    0.6631          0.6870  
      lgbm_full_binary             0.5693          0.6847  
      nrms_g10                     0.5949          0.6184  
      content_smoothed_ctr         0.3795          0.6162  
      candidate_smoothed_ctr       0.3017          0.6166  
      category_ctr                 0.3380          0.4984  
      popularity                   0.0309          0.4897  
      random                       0.2814          0.4996  
test  deep_hybrid                  0.6464          0.6660  
      lgbm_full                    0.6237          0.6727  
      lgbm_full_binary             0.5114          0.6661  
      nrms_g10                     0.5926          0.5788  
      content_smoothed_ctr         0.2784          0.5952  
      candidate_smoothed_ctr       0.2593          0.5953  
      category_ctr                 0.3059          0.4808  
      popularity                   0.0287          0.4666  
      random                       0.2378          0.4844

deep_hybrid − other              95% CI
split vs        metric                                                 
val   nrms_g10  ndcg@10                      0.0680  [+0.0648, +0.0713]
                cold_recall@10               0.1071  [+0.0737, +0.1398]
      lgbm_full ndcg@10                     -0.0024  [-0.0042, -0.0005]
                cold_recall@10               0.0368  [+0.0139, +0.0613]
test  nrms_g10  ndcg@10                      0.0840  [+0.0808, +0.0874]
                cold_recall@10               0.0526  [+0.0183, +0.0858]
      lgbm_full ndcg@10                     -0.0028  [-0.0047, -0.0008]
                cold_recall@10               0.0214  [-0.0025, +0.0452]

## 7. Looking inside: what the attention picks out

On test impressions that contain a click, with the clicked article as the candidate, compare Group 10's candidate-independent attention with the new candidate-aware attention:

* **Concentration.** Weight on the top history item, against the uniform 1/(history length).
* **Relevance.** How often the most-attended history article shares the clicked article's category, against a randomly chosen history article. Attention that tracks what the user is about to read should beat random clearly.

In [7]:
news_raw = pd.read_csv(os.path.join(RAW_DIR, "news.tsv"), sep="\t", header=None, usecols=[0, 1, 3],
                       names=["news_id", "category", "title"])
cat_of_idx = news_raw["category"].to_numpy()
te_split = inputs["splits"]["test"]
clicked = pd.DataFrame({"imp": te_split["imp"], "news": te_split["news"], "label": te_split["label"]})
clicked = clicked[clicked["label"] == 1].drop_duplicates("imp").sample(frac=1.0, random_state=SEED).head(20000)
w_g10, hist = H.history_attention(model, clicked["imp"].to_numpy())
w_cand = H.candidate_attention(model, clicked["imp"].to_numpy(), clicked["news"].to_numpy())
n_hist = (~model.hist_mask[torch.as_tensor(clicked["imp"].to_numpy())].cpu().numpy()).sum(1)
has = n_hist >= 2
target_cat = cat_of_idx[clicked["news"].to_numpy()[has]]
rng = np.random.default_rng(SEED)
rand = np.array([row_h[rng.choice(np.flatnonzero(row_w > 0))] for row_h, row_w in zip(hist[has], w_cand[has])])

def describe(w):
    top = hist[has][np.arange(has.sum()), w[has].argmax(1)]
    return {"mean weight on top history item": float(w[has].max(1).mean()),
            "top-attended item shares clicked article's category": float((cat_of_idx[top] == target_cat).mean())}

summary = pd.DataFrame({"Group 10 attention (fixed query)": describe(w_g10), "candidate-aware attention": describe(w_cand)})
summary.loc["uniform weight 1/n (reference)"] = float((1 / n_hist[has]).mean())
summary.loc["random history item shares category (reference)"] = float((cat_of_idx[rand] == target_cat).mean())
display(summary.round(4))
print(f"{int(has.sum()):,} test impressions with at least 2 history items")

# one worked example: a user with 10 past clicks and a clicked article whose category appears in that history
ok = np.flatnonzero(has & (n_hist == 10))
i = next(int(j) for j in ok if (cat_of_idx[hist[j]] == cat_of_idx[clicked["news"].iloc[j]]).any())
print("clicked:", news_raw["title"].iloc[clicked["news"].iloc[i]], f"[{cat_of_idx[clicked['news'].iloc[i]]}]")
pd.DataFrame({"candidate-aware weight": w_cand[i], "Group 10 weight": w_g10[i],
              "history article": news_raw["title"].to_numpy()[hist[i]], "category": cat_of_idx[hist[i]]}
             ).sort_values("candidate-aware weight", ascending=False).round(3)

,Group 10 attention (fixed query),candidate-aware attention
mean weight on top history item,0.1638,0.1332
top-attended item shares clicked article's category,0.1968,0.1833
uniform weight 1/n (reference),0.1332,0.1332
random history item shares category (reference),0.2024,0.2024


19,190 test impressions with at least 2 history items
clicked: Kiss Cancel 'End of the Road' Tour of Australia and New Zealand Due to Illness [music]


,candidate-aware weight,Group 10 weight,history article,category
9,0.1,0.079,Previewing every key matchup in LSU-Alabama sh...,sports
5,0.1,0.082,Lane Kiffin interested in Florida State head c...,sports
4,0.1,0.128,SR-71 Blackbird: The Cold War spy plane that's...,news
2,0.1,0.117,McDonald's CEO pushed out after relationship w...,finance
6,0.1,0.090,"What Tom Brady, Lamar Jackson Told Each Other ...",sports
1,0.1,0.091,Bette Midler mocks Lori Loughlin's college adm...,music
0,0.1,0.080,New video of El Chapo's son being arrested,video
7,0.1,0.112,Reports: LSU LB Michael Divinity removed from ...,sports
8,0.1,0.100,One of FBI's Most Wanted fugitives offers surr...,news
3,0.1,0.121,Why FSU fired football coach Willie Taggart wi...,sports


## 8. What step 4 shows

* **The main model beats every baseline and Group 10's network.** Test nDCG@10 is 0.428, against 0.357 for the best baseline and 0.344 for Group 10's network (+0.084, 95% CI [+0.081, +0.087]). Adding behavioural and matching evidence to Group 10's content network closes the gap diagnosed in step 3b.
* **Against LightGBM it gives up a little overall ranking for better cold recall.** nDCG@10 is 0.003 lower on both splits (intervals just below zero). Cold Recall@10 is higher: +0.037 on validation (CI [+0.014, +0.061]) and +0.021 on test (CI [−0.003, +0.045]), so the cold gain is clear on validation and points the same way on test without reaching significance.
* **Training converges quickly.** The best validation epoch is 3, and training stops at epoch 6. Train nDCG@10 is 0.514 against 0.451 on validation. Validation Cold Recall@10 swings between 0.60 and 0.72 from epoch to epoch, so cold-recall differences of a few points between runs should be read with care.
* **Neither attention singles out relevant past clicks.** Group 10's attention puts 16% of its weight on its top item (uniform: 13%); in this run it did not collapse, unlike the first run (81%). The candidate-aware attention stayed at uniform weights, so in effect it averages the user's last 10 clicks and matches that average against each candidate. In both, the most-attended click shares the clicked article's category slightly less often than a random click (19.7% and 18.3% against 20.2%). The personalisation the model uses comes from the average taste profile and the engineered matching features, not from picking out particular past clicks. Step 5 measures how much the candidate-aware branch adds (`no_candidate_attention`).

*Based on the run with feature schema v0.1 and the interim evaluator. The final numbers and intervals come from step 6.*